In [ ]:
# BLOCK 0 - Dependencies
%pip install dicom2nifti pydicom

In [ ]:
import os, shutil, tempfile
from collections import defaultdict
import numpy as np
import pydicom
import dicom2nifti
import dicom2nifti.settings as settings
from dicom2nifti.convert_dicom import dicom_series_to_nifti

# ─── EDIT THESE TWO PATHS ───────────────────────────────────────────────────────
DICOM_ROOT = r"PATH_TO_DICOMS"
NIFTI_ROOT = r"PATH_TO_NIFTI_OUTPUT"
# ────────────────────────────────────────────────────────────────────────────────

settings.reorient_nifti = True
settings.disable_validate_slice_orientation = True
settings.disable_validate_slice_increment  = True

os.makedirs(NIFTI_ROOT, exist_ok=True)

ORIENTATION_TOL = 1e-3
SPACING_TOL     = 1.0


def _orientation_key(ds):
    iop = ds.get("ImageOrientationPatient")
    return None if not iop or len(iop) != 6 else tuple(round(float(v), 3) for v in iop)


def _z_coord(ds):
    ipp = np.array(ds.ImagePositionPatient, float)
    iop = np.array(ds.ImageOrientationPatient, float).reshape(2, 3)
    return np.dot(ipp, np.cross(iop[0], iop[1]))


def group_by_series(folder):
    series = defaultdict(list)
    for root, _, files in os.walk(folder):
        for fn in files:
            if fn.lower().endswith(".dcm"):
                fp = os.path.join(root, fn)
                try:
                    uid = pydicom.dcmread(fp, stop_before_pixels=True,
                                          specific_tags=["SeriesInstanceUID"]).SeriesInstanceUID
                    series[uid].append(fp)
                except Exception:
                    pass
    return series


def pick_target_series(series_dict):
    best = []
    for files in series_dict.values():
        if len(files) <= 50:
            continue
        try:
            t = pydicom.dcmread(files[0], stop_before_pixels=True,
                                specific_tags=["ImageType"]).ImageType
            if isinstance(t, str):
                t = [s.strip().upper() for s in t.split("\\")]
        except Exception:
            continue
        if {"ORIGINAL", "PRIMARY", "AXIAL"}.issubset(t) and len(files) > len(best):
            best = files
    return best


def filter_by_orientation(files):
    groups = defaultdict(list)
    for fp in files:
        try:
            key = _orientation_key(pydicom.dcmread(fp, stop_before_pixels=True,
                                                   specific_tags=["ImageOrientationPatient"]))
            if key:
                groups[key].append(fp)
        except Exception:
            pass
    return max(groups.values(), key=len) if groups else []


def filter_by_spacing_and_block(files):
    if len(files) < 3:
        return files
    dsets = {fp: pydicom.dcmread(fp, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient",
                                                "ImageOrientationPatient"]) for fp in files}
    z_vals = {fp: _z_coord(ds) for fp, ds in dsets.items()}
    ordered = sorted(z_vals, key=z_vals.get)
    spacings = [abs(z_vals[ordered[i+1]] - z_vals[ordered[i]])
                for i in range(len(ordered)-1)]
    if not spacings:
        return files
    target = np.median(spacings)

    blocks, cur = [], [ordered[0]]
    for i in range(1, len(ordered)):
        if abs(abs(z_vals[ordered[i]] - z_vals[ordered[i-1]]) - target) <= SPACING_TOL:
            cur.append(ordered[i])
        else:
            blocks.append(cur)
            cur = [ordered[i]]
    blocks.append(cur)
    return max(blocks, key=len)


for exam in os.listdir(DICOM_ROOT):
    exam_path = os.path.join(DICOM_ROOT, exam)
    if not os.path.isdir(exam_path):
        continue

    out_nii = os.path.join(NIFTI_ROOT, f"{exam}.nii.gz")
    if os.path.exists(out_nii):
        print(f"{exam}: skipped (already converted)")
        continue

    series_files = pick_target_series(group_by_series(exam_path))
    if not series_files:
        print(f"{exam}: error (no suitable series)")
        continue

    orient_ok = filter_by_orientation(series_files)
    space_ok  = filter_by_spacing_and_block(orient_ok)

    if len(space_ok) < 50:
        print(f"{exam}: error (too few valid slices)")
        continue

    print(f"{exam}: converting …")                      # ← new line

    try:
        with tempfile.TemporaryDirectory() as tmp:
            for fp in space_ok:
                shutil.copy2(fp, tmp)
            dicom_series_to_nifti(tmp, out_nii, reorient_nifti=True)
        print(f"{exam}: done ({len(space_ok)} slices)")
    except Exception as e:
        print(f"{exam}: error ({e})")

print("All requested conversions finished.")